In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# Pathsクラス
class Paths:
    p = "/Users/shirokoshikentaro/Downloads/house-prices-advanced-regression-techniques/"
    train = p + "train.csv"
    test = p + "test.csv"
    sample = p + "sample_submission.csv"

# === データ読み込み ===
print("データ読み込み中...")
train_raw = pd.read_csv(Paths.train)
test_raw = pd.read_csv(Paths.test)

# === 新規特徴量の作成 ===
print("新規特徴量を作成中...")

# 1. 品質×面積（最強の組み合わせ！）
train_raw["QualGrLiv"] = train_raw["OverallQual"] * train_raw["GrLivArea"]
test_raw["QualGrLiv"] = test_raw["OverallQual"] * test_raw["GrLivArea"]

# 2. 総面積
train_raw["TotalSF"] = train_raw["TotalBsmtSF"] + train_raw["1stFlrSF"] + train_raw["2ndFlrSF"]
test_raw["TotalSF"] = test_raw["TotalBsmtSF"] + test_raw["1stFlrSF"] + test_raw["2ndFlrSF"]

# 3. 品質×総面積
train_raw["QualTotalSF"] = train_raw["OverallQual"] * train_raw["TotalSF"]
test_raw["QualTotalSF"] = test_raw["OverallQual"] * test_raw["TotalSF"]

# 4. 築年数
train_raw["Age"] = train_raw["YrSold"] - train_raw["YearBuilt"]
test_raw["Age"] = test_raw["YrSold"] - test_raw["YearBuilt"]

# 5. リフォーム経過年数
train_raw["RemodAge"] = train_raw["YrSold"] - train_raw["YearRemodAdd"]
test_raw["RemodAge"] = test_raw["YrSold"] - test_raw["YearRemodAdd"]

# 6. 総バスルーム数
train_raw["TotalBath"] = (train_raw["FullBath"] + 0.5*train_raw["HalfBath"].fillna(0) + 
                          train_raw["BsmtFullBath"].fillna(0) + 0.5*train_raw["BsmtHalfBath"].fillna(0))
test_raw["TotalBath"] = (test_raw["FullBath"] + 0.5*test_raw["HalfBath"].fillna(0) + 
                         test_raw["BsmtFullBath"].fillna(0) + 0.5*test_raw["BsmtHalfBath"].fillna(0))

# 7. 1部屋あたりの面積
train_raw["AreaPerRoom"] = train_raw["GrLivArea"] / train_raw["TotRmsAbvGrd"].replace(0, 1)
test_raw["AreaPerRoom"] = test_raw["GrLivArea"] / test_raw["TotRmsAbvGrd"].replace(0, 1)

# 8. ガレージ指数
train_raw["GarageScore"] = train_raw["GarageCars"] * train_raw["GarageArea"]
test_raw["GarageScore"] = test_raw["GarageCars"] * test_raw["GarageArea"]

# === 特徴量定義（元の14個 + 新規8個 = 22個）===
numeric_features = [
    # 元の特徴量
    "LotArea", "YearBuilt", "YearRemodAdd",
    "GrLivArea", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "FullBath", "BedroomAbvGr", "TotRmsAbvGrd",
    "GarageCars", "GarageArea",
    "OverallQual", "OverallCond",
    # 新規特徴量
    "QualGrLiv", "TotalSF", "QualTotalSF", 
    "Age", "RemodAge", "TotalBath", 
    "AreaPerRoom", "GarageScore"
]

categorical_features = [
    "Neighborhood", "BldgType", "HouseStyle",
    "MSZoning", "Foundation", "GarageType"
]

features = numeric_features + categorical_features
print(f"\n使用する特徴量: {len(features)}個")
print(f"  数値特徴量: {len(numeric_features)}個（元14個 + 新規8個）")
print(f"  カテゴリ特徴量: {len(categorical_features)}個")

# === 新規特徴量の相関を確認 ===
print("\n新規特徴量のSalePriceとの相関:")
new_features = ["QualGrLiv", "TotalSF", "QualTotalSF", "Age", "RemodAge", "TotalBath", "AreaPerRoom", "GarageScore"]
for feature in new_features:
    corr = train_raw[[feature, "SalePrice"]].corr().iloc[0, 1]
    print(f"  {feature:20s}: {corr:6.3f}")

# === 特徴量を抽出 ===
train_features = train_raw[features].copy()
test_features = test_raw[features].copy()

# === 欠損値処理 ===
print("\n欠損値処理中...")
for col in numeric_features:
    train_features[col] = train_features[col].fillna(0)
    test_features[col] = test_features[col].fillna(0)

for col in categorical_features:
    train_features[col] = train_features[col].fillna("None")
    test_features[col] = test_features[col].fillna("None")

# === カテゴリ変数のエンコーディング ===
print("カテゴリ変数をエンコーディング中...")
train_encoded = {}
test_encoded = {}

for col in numeric_features:
    train_encoded[col] = train_features[col].values
    test_encoded[col] = test_features[col].values

for col in categorical_features:
    all_data = pd.concat([
        train_features[col].astype(str), 
        test_features[col].astype(str)
    ], axis=0)
    
    le = LabelEncoder()
    le.fit(all_data)
    
    train_encoded[col] = le.transform(train_features[col].astype(str))
    test_encoded[col] = le.transform(test_features[col].astype(str))

# === 新しいDataFrameを作成 ===
X_train = pd.DataFrame(train_encoded, columns=features)
X_test = pd.DataFrame(test_encoded, columns=features)
y_train = train_raw["SalePrice"].copy()  # 対数変換なし

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

# === モデルパラメータ ===
params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 16,
    "learning_rate": 0.1,
    "n_estimators": 10000,
    "random_state": 123,
    "verbose": -1,
}

# === CV実行 ===
print("\n" + "=" * 60)
print("Cross Validation 開始")
print("=" * 60)

n_splits = 5
cv = KFold(n_splits=n_splits, shuffle=True, random_state=123)
metrics = []

for nfold, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    print(f"\n{'=' * 10} Fold {nfold} {'=' * 10}")
    X_tr, y_tr = X_train.iloc[train_index], y_train.iloc[train_index]
    X_val, y_val = X_train.iloc[valid_index], y_train.iloc[valid_index]

    model_fold = lgb.LGBMRegressor(**params)
    model_fold.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(0),
        ],
    )

    y_tr_pred = model_fold.predict(X_tr)
    y_val_pred = model_fold.predict(X_val)
    
    tr_rmse = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    
    metrics.append([nfold, tr_rmse, val_rmse])
    print(f"Train RMSE: {tr_rmse:,.0f}, Valid RMSE: {val_rmse:,.0f}")

# === CVスコア ===
metrics_array = np.array(metrics)
print("\n" + "=" * 60)
print("CV結果")
print("=" * 60)
print("[CV] train: {:.0f}±{:.0f}, valid: {:.0f}±{:.0f}".format(
    metrics_array[:,1].mean(), metrics_array[:,1].std(),
    metrics_array[:,2].mean(), metrics_array[:,2].std(),
))
print("=" * 60)

# === 全データで最終モデルを学習 ===
print("\n全データでモデルを再学習中...")
model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(100),
    ],
)
print("✅ 再学習完了！")

# === testデータで予測 ===
print("\n予測実行中...")
y_test_pred = model.predict(X_test)

print(f"\n予測価格:")
print(f"  平均: ${y_test_pred.mean():,.0f}")
print(f"  範囲: ${y_test_pred.min():,.0f} ~ ${y_test_pred.max():,.0f}")

# === 提出ファイル ===
df_submit = pd.DataFrame({
    "Id": test_raw["Id"],
    "SalePrice": y_test_pred,
})

print("\n提出ファイル:")
print(df_submit.head(10))
print(f"\nデータ数: {len(df_submit)}")

df_submit.to_csv("submission_with_new_features.csv", index=False)
print("\n✅ submission_with_new_features.csv を保存しました！")